# LABORATORIO 2 – ANALIZADOR SINTÁCTICO PARA MINIJAVA

**Asignatura:** Compiladores  
**Estudiante:** Maria Alejandra Arias Medina  
**NRC:** 2439  
En este laboratorio desarrollé un analizador sintáctico para un subconjunto del lenguaje MiniJava. La idea es leer un archivo de texto, revisar si las instrucciones cumplen las reglas que definí y mostrar la estructura que se reconoce. También incluí algunos casos de recuperación para que, cuando aparezcan ciertos errores, el análisis pueda continuar y señalar el problema.

>

# Gramática libre de contexto de MiniJava

## 1. Definición 

Para establecer qué formas de instrucciones puede reconocer el programa, definí una gramática libre de contexto. Esta gramática se expresa mediante la cuádrupla:

\[
G=(N,T,P,S)
\]

Cada elemento tiene una función dentro de la descripción del lenguaje:
- N: reúne los símbolos que representan las estructuras que se van a desarrollar.
- T: contiene los elementos concretos que aparecen en el código, como palabras reservadas, operadores y delimitadores.
- P: incluye las reglas que indican cómo se pueden combinar esos símbolos.
- S: señala la regla por la que empieza el análisis.
Para este caso tomé como símbolo inicial \(S=Programa\). Desde allí se puede reconocer una secuencia de instrucciones, ya sea que estén escritas directamente o agrupadas dentro de un bloque. La barra \(\mid\) indica que hay más de una alternativa y \(\varepsilon\) representa una parte opcional o vacía.

La gramática describe el subconjunto de MiniJava que se trabaja en este laboratorio; no pretende cubrir todas las características del lenguaje Java.

## 2. Simbolos terminales 

Los terminales son los elementos que sí se escriben en el programa MiniJava. Para que sea más fácil consultarlos, los separé por grupos. Los identificadores y literales se muestran como categorías, ya que su contenido puede cambiar de una instrucción a otra.
| Categoría | Elementos que reconoce la gramática |
|---|---|
| Tipos de datos | `int`, `float`, `boolean`, `bool`, `char`, `String`, `string` |
| Identificadores y valores | `ID`, `INT_LITERAL`, `FLOAT_LITERAL`, `STRING_LITERAL`, `CHAR_LITERAL`, `true`, `false` |
| Control de flujo | `if`, `else`, `while`, `do`, `for`, `switch`, `case`, `default` |
| Salida y control de ejecución | `System`, `out`, `print`, `println`, `break`, `continue` |
| Asignación | `=`, `+=`, `-=`, `*=`, `/=` |
| Operaciones aritméticas | `+`, `-`, `*`, `/`, `%`, `++`, `--` |
| Comparación y lógica | `<`, `<=`, `>`, `>=`, `==`, `!=`, `&&`, `||`, `!` |
| Signos de agrupación y separación | `(`, `)`, `{`, `}`, `;`, `:`, `,` |

Los espacios, tabulaciones y saltos de línea sirven para separar el código, pero no se consideran terminales de la gramática. De igual manera, los comentarios se ignoran antes de realizar el análisis sintáctico.


## 3. Simbolos no terminales e inicial 
Los no terminales son nombres que utilicé para organizar las diferentes partes de una instrucción. No aparecen necesariamente escritos de esa forma en el archivo de entrada, porque funcionan como etiquetas para construir las reglas.

Símbolo de partida: \(S=Programa\).

Organicé los no terminales de acuerdo con la función que cumplen:
- Estructura general: \(Programa\), \(ListaSentencias\), \(Sentencia\) y \(Bloque\).
- Instrucciones básicas: \(Tipo\), \(Declaracion\), \(Asignacion\), \(OperadorAsignacion\), \(Incremento\), \(Impresion\) y \(Salto\).
- Decisiones: \(Condicional\) y \(Alternativa\), que permiten representar las ramas de un if.
- Repeticiones: \(CicloWhile\), \(CicloDoWhile\), \(CicloFor\), \(InicioFor\) y \(PasoFor\).
- Selección por casos: \(Seleccion\), \(ListaCasos\), \(Caso\), \(OpcionDefault\) y \(LiteralCaso\).
- Construcción de expresiones: \(Expresion\), \(ExpresionOR\), \(ExpresionAND\), \(ExpresionRelacional\), \(ExpresionAditiva\), \(ExpresionMultiplicativa\), \(ExpresionUnaria\), \(Primario\) y \(Literal\).
Esta división ayuda a que cada regla tenga una responsabilidad concreta. Por ejemplo, CicloFor organiza la estructura del ciclo, mientras que Expresion se encarga de la condición o de los valores que se utilizan en sus partes.


## 4. Producciones
En esta parte presento las reglas principales que utilicé para describir las instrucciones. En las producciones, los nombres como ID o INT_LITERAL representan las categorías de elementos que puede encontrar el analizador.
   ## 4.1. Sentencias simples 
   Aquí se encuentran las instrucciones que normalmente se escriben en una sola línea: declarar una variable, asignarle un dato, cambiar su valor o mostrar información. También se incluyen break y continue.

   \[
\begin{aligned}
Tipo &\rightarrow int\mid float\mid boolean\mid bool\mid char\mid String\mid string\\
Declaracion &\rightarrow Tipo\ ID\ [=\ Expresion]\ ;\\
Asignacion &\rightarrow ID\ OperadorAsignacion\ Expresion\ ;\\
OperadorAsignacion &\rightarrow =\mid +=\mid -=\mid *=\mid /=\\
Incremento &\rightarrow ID\ ( ++\mid -- )\ ;\\
Impresion &\rightarrow System.out.(print\mid println)(Expresion)\ ;\\
Salto &\rightarrow break\ ;\mid continue\ ;\\
\end{aligned}
\]


   ## 4.2. Programa y bloques 
  El programa comienza con una lista de sentencias. Esa lista puede contener varias instrucciones o estar vacía. Un bloque permite reunir sentencias entre llaves.

  \[
\begin{aligned}
Programa &\rightarrow ListaSentencias \\
ListaSentencias &\rightarrow Sentencia\ ListaSentencias\mid\varepsilon \\
Sentencia &\rightarrow Declaracion\mid Asignacion\mid Incremento\mid Impresion\mid Salto\\
&\mid Condicional\mid CicloWhile\mid CicloDoWhile\mid CicloFor\mid Seleccion\mid Bloque\\
Bloque &\rightarrow \{\ ListaSentencias\ \}\\
\end{aligned}
\]

   ## 4.3. Ciclos (while, do-while, for)
   Las siguientes reglas representan las tres estructuras de repetición contempladas. En el ciclo for se distinguen sus partes para que la inicialización, la condición y el paso queden separados.
\[
\begin{aligned}
CicloWhile &\rightarrow while\ (Expresion)\ Bloque\\
CicloDoWhile &\rightarrow do\ Bloque\ while\ (Expresion)\ ;\\
CicloFor &\rightarrow for\ (InicioFor\ ;\ Expresion\ ;\ [PasoFor])\ Bloque\\
InicioFor &\rightarrow Tipo\ ID\ =\ Expresion\mid ID\ =\ Expresion\\
PasoFor &\rightarrow ID\ ( ++\mid -- )\mid ID\ OperadorAsignacion\ Expresion\\
\end{aligned}
\]

   ## 4.4. Expresiones (jerarquía de precedencia)
Para analizar expresiones no basta con reconocer los operadores: también es necesario respetar el orden en que se agrupan. Por eso separé las expresiones en niveles. Los operadores de multiplicación se resuelven antes que los de suma; después se consideran las comparaciones y, finalmente, los operadores lógicos.
\[
\begin{aligned}
Expresion &\rightarrow ExpresionOR\\
ExpresionOR &\rightarrow ExpresionAND\ (||\ ExpresionAND)^*\\
ExpresionAND &\rightarrow ExpresionRelacional\ (&&\ ExpresionRelacional)^*\\
ExpresionRelacional &\rightarrow ExpresionAditiva\ ((<|<=|>|>=|==|!=)\ ExpresionAditiva)^*\\
ExpresionAditiva &\rightarrow ExpresionMultiplicativa\ ((+|-)\ ExpresionMultiplicativa)^*\\
ExpresionMultiplicativa &\rightarrow ExpresionUnaria\ ((*|/|%)\ ExpresionUnaria)^*\\
ExpresionUnaria &\rightarrow (!|-)\ ExpresionUnaria\mid Primario\\
Primario &\rightarrow ID\mid Literal\mid (Expresion)\\
Literal &\rightarrow INT\_LITERAL\mid FLOAT\_LITERAL\mid STRING\_LITERAL\\
&\mid CHAR\_LITERAL\mid true\mid false\\
\end{aligned}
\]
El uso de estos niveles permite que una expresión como 4 + 2 * 3 se agrupe como una suma cuyo segundo elemento es una multiplicación, y no como una operación realizada de izquierda a derecha sin tener en cuenta la precedencia.

   ## 4.5. Condicional (if / else if / else)
La estructura condicional permite ejecutar un bloque dependiendo del resultado de una expresión. La parte alternativa puede omitirse, contener un else o incluir otro if, que es la forma en que se representa una cadena else if.
\[
\begin{aligned}
Condicional &\rightarrow if\ (Expresion)\ Bloque\ Alternativa\\
Alternativa &\rightarrow else\ Bloque\mid else\ Condicional\mid\varepsilon\\
\end{aligned}
\]

   ## 4.6. Selección múltiple (switch-case)
La instrucción switch evalúa una expresión y permite comparar su resultado con diferentes alternativas case. También se puede incluir default para representar el caso que no coincide con las alternativas anteriores.
\[
\begin{aligned}
Seleccion &\rightarrow switch\ (Expresion)\ \{\ ListaCasos\ [OpcionDefault]\ \}\\
ListaCasos &\rightarrow Caso\ ListaCasos\mid\varepsilon\\
Caso &\rightarrow case\ LiteralCaso\ :\ ListaSentencias\\
OpcionDefault &\rightarrow default\ :\ ListaSentencias\\
LiteralCaso &\rightarrow Literal\mid -INT\_LITERAL\mid -FLOAT\_LITERAL\\
\end{aligned}
\]

## 5. Resolución del *dangling else*

El llamado else colgante aparece cuando hay un condicional dentro de otro y no queda claro a cuál if pertenece el else. Para evitar esa ambigüedad se aplica la convención habitual: el else se relaciona con el if pendiente más cercano.

En las reglas que utilicé, la alternativa puede contener otro Condicional. Así se pueden representar las cadenas if, else if y else, y mantener la asociación del else con el condicional que todavía no tiene alternativa.

## 6. Observaciones
La gramática y el programa deben entenderse como partes de una misma implementación. Las producciones anteriores resumen las construcciones que se contemplan en este ejercicio y se relacionan con las reglas definidas en la celda de código. No es un analizador de Java completo: trabaja con los tipos, expresiones y sentencias que se incluyeron para el laboratorio.

# Manual de uso del analizador 

## 1. Descripción

Este notebook permite revisar la estructura de un programa escrito con las instrucciones del subconjunto MiniJava definido en la gramática. El analizador está construido en Python con la librería Pyparsing y toma como entrada un archivo de texto con extensión .txt.

Al procesar el archivo, el programa intenta reconocer las sentencias, informa si encontró problemas de sintaxis y construye un árbol con la estructura que pudo identificar. Para algunos errores que se tuvieron en cuenta durante la implementación, también intenta recuperarse y continuar con las instrucciones siguientes.

El objetivo es revisar la forma del código. El notebook no ejecuta las instrucciones ni determina si el resultado del programa es correcto desde el punto de vista lógico.

## 2. Requisitos

Antes de iniciar, se debe contar con lo siguiente:
- Python 3 instalado o acceso a Google Colab.
- La librería Pyparsing disponible en el entorno.
- El notebook del laboratorio.
- Un archivo .txt que contenga el código que se quiere revisar.
- Conexión a internet si se va a instalar la librería desde Google Colab.

## 3. Librerias

La herramienta principal es Pyparsing, que permite construir las expresiones y reglas de la gramática directamente en Python. También se utiliza re para apoyar algunas búsquedas mediante expresiones regulares y sys para configurar aspectos del entorno de ejecución.

Si Pyparsing no está instalado, se puede agregar desde una celda de Colab con:
!pip install pyparsing

En un equipo local, la instalación se puede realizar desde la terminal:
python -m pip install pyparsing

La solución no depende de generadores externos de analizadores como PLY, ANTLR o Lark; las reglas se construyen usando Pyparsing.

## 4. Estructura del Notebook

El archivo está organizado en cuatro celdas para que sea más sencillo seguir el desarrollo:
1. Presentación del laboratorio: incluye el nombre del trabajo y los datos de la entrega individual.
2. Gramática de MiniJava: explica la definición formal, los símbolos utilizados y las producciones de las instrucciones.
3. Manual de uso: reúne los requisitos, los pasos para ejecutar el análisis y la forma de interpretar los resultados.
4. Implementación: contiene el código Python, la definición de las reglas, la construcción de la representación sintáctica, el manejo de algunos errores y la lectura del archivo de entrada.

## 5. Guia para ejecutar

1. Abrir el archivo .ipynb en Google Colab o en Jupyter Notebook.
2. Revisar que el entorno tenga instalada la librería Pyparsing. Si hace falta, ejecutar el comando de instalación indicado en la sección anterior.
3. Ejecutar las celdas en orden, desde la primera hasta la última. En Colab también se puede utilizar la opción Entorno de ejecución → Ejecutar todas.
4. Al llegar a la parte de carga, seleccionar el archivo .txt que contiene el programa que se desea analizar.
5. Esperar a que termine el proceso y revisar el resultado que aparece debajo de la celda.
6. Si se informa algún error, comparar la línea señalada con el contenido del archivo y revisar también el árbol sintáctico.
Para analizar otro programa, se puede volver a ejecutar la parte de carga y escoger un archivo diferente. Es importante que el documento sea de texto plano y que tenga la extensión .txt.

## 6. Formato del archivo de entrada 

El código que se va a revisar debe guardarse en un archivo de texto plano. Se pueden escribir varias instrucciones y utilizar bloques con llaves para agruparlas. Por ejemplo:

int contador = 0;
while (contador < 3) {
    contador++;
    System.out.println(contador);
}

Al preparar el archivo conviene comprobar que los paréntesis, las llaves, los dos puntos y los puntos y coma estén ubicados según la instrucción. En la carpeta de pruebas se incluyen ejemplos para probar diferentes situaciones.

## 7. Árbol de análisis sintáctico
Cuando el analizador reconoce una construcción, la organiza en una estructura de listas anidadas. Cada lista representa una parte del programa y puede contener otras listas para mostrar los elementos que dependen de ella. Por ejemplo, una estructura condicional incluye la condición y el bloque asociado; una expresión puede contener los operandos y operadores que la forman.

Esta representación se conoce como árbol sintáctico y sirve para observar cómo se relacionan las partes del código. No corresponde a la ejecución del programa: muestra la organización que el analizador reconoció.

Cuando se identifica un error que cuenta con recuperación, se puede agregar al árbol un elemento con la estructura general:
['error', contexto, descripcion]
El contexto y la descripción dependen del problema que se haya detectado.

## 8. Errores sintácticos
El código contempla mecanismos para señalar algunos errores frecuentes, por ejemplo, cuando falta un separador o un signo de cierre en determinadas instrucciones, o cuando una cabecera de control está incompleta. En los casos que tienen una regla de recuperación, el analizador registra el problema e intenta continuar desde una parte posterior del archivo.

La recuperación depende de los casos que fueron programados. Por eso, no se debe asumir que cualquier error posible será corregido automáticamente. Si aparece una instrucción que no pertenece al subconjunto definido o un error que no está contemplado, puede que el analizador no consiga continuar de la manera esperada.

## 9. Limitaciones
- El analizador reconoce un subconjunto de MiniJava, no toda la sintaxis de Java.
- Su función es sintáctica: no ejecuta el código ni comprueba que los cálculos produzcan el resultado esperado.
- No realiza un análisis semántico completo; por ejemplo, no verifica todos los usos de variables ni la compatibilidad entre tipos.
- La recuperación está limitada a los errores que se contemplaron en la implementación.
- Los resultados deben revisarse junto con el archivo de entrada, especialmente cuando se presentan varios errores en una misma estructura.

In [ ]:
import pyparsing as pp
import re
pp.ParserElement.enable_packrat()
pp.ParserElement.set_default_whitespace_chars(" \t\r\n")

class Nodo:
    def __init__(self, valor): self.valor = valor
    def __repr__(self): return repr(self.valor)

def valor(x):
    if isinstance(x, Nodo): return valor(x.valor)
    if isinstance(x, pp.ParseResults): return [valor(v) for v in x]
    if isinstance(x, list): return [valor(v) for v in x]
    return x

def lista(x):
    return valor(x) if isinstance(x, (Nodo, pp.ParseResults)) else [valor(v) for v in x]

TIPOS = ("int", "float", "boolean", "bool", "char", "String", "string")
kw = lambda s: pp.Keyword(s)
COMENTARIOS = pp.Regex(r"//[^\n]*|/\*[\s\S]*?\*/")

ident = pp.Regex(r"[A-Za-z][A-Za-z0-9]*")
numero = pp.Regex(r"(?:\d+\.\d+(?:[eE][+-]?\d+)?|\d+[eE][+-]?\d+|\d+)")
string_lit = pp.Regex(r'"(?:[^"\\\n]|\\.)*"')
char_lit = pp.Regex(r"'(?:[^'\\\n]|\\.)'")
boolean_lit = pp.MatchFirst([kw("true"), kw("false")])
atom = pp.MatchFirst([numero, string_lit, char_lit, boolean_lit, ident])
expr = pp.Forward()
parens = pp.Suppress("(") + expr + pp.Suppress(")")
base = pp.MatchFirst([parens, atom])
def expr_unaria(t): return Nodo([valor(t[0][0]), valor(t[0][1])])
def expr_binaria(t): return Nodo(lista(t[0]))
expr <<= pp.infix_notation(base, [
    (pp.MatchFirst([pp.Literal("!"), pp.Literal("-")]), 1, pp.OpAssoc.RIGHT, expr_unaria),
    (pp.MatchFirst([pp.Literal("*"), pp.Literal("/"), pp.Literal("%")]), 2, pp.OpAssoc.LEFT, expr_binaria),
    (pp.MatchFirst([pp.Literal("+"), pp.Literal("-")]), 2, pp.OpAssoc.LEFT, expr_binaria),
    (pp.MatchFirst([pp.Literal("<="), pp.Literal(">="), pp.Literal("<"), pp.Literal(">"), pp.Literal("=="), pp.Literal("!=")]), 2, pp.OpAssoc.LEFT, expr_binaria),
    (pp.Literal("&&"), 2, pp.OpAssoc.LEFT, expr_binaria),
    (pp.Literal("||"), 2, pp.OpAssoc.LEFT, expr_binaria),
])
expr.set_parse_action(lambda t: Nodo(valor(t[0])))

LP, RP = pp.Suppress("("), pp.Suppress(")")
LB, RB = pp.Suppress("{"), pp.Suppress("}")
SEMI, COLON = pp.Suppress(";"), pp.Suppress(":")
tipo = pp.MatchFirst([kw(t) for t in TIPOS])

sentencia = pp.Forward()
bloque = pp.Forward()
declaracion = (tipo + ident + pp.Optional(pp.Suppress("=") + expr) + SEMI)
def pa_decl(t):
    r = [t[0], t[1]]
    if len(t) == 3:
        e = valor(t[2]); r += e if isinstance(e, list) else [e]
    return Nodo(r)
declaracion.set_parse_action(pa_decl)

op_asig = pp.MatchFirst([pp.Literal(x) for x in ("+=", "-=", "*=", "/=", "=")])
asignacion = (ident + op_asig + expr + SEMI)
def pa_asig(t):
    e = valor(t[2]); return Nodo([t[0], t[1]] + (e if isinstance(e, list) else [e]))
asignacion.set_parse_action(pa_asig)
incremento = (ident + pp.MatchFirst([pp.Literal("++"), pp.Literal("--")]) + SEMI).set_parse_action(lambda t: Nodo([t[0], t[1]]))
impresion_java = (pp.Suppress(kw("System")) + pp.Suppress(".") + pp.Suppress(kw("out")) + pp.Suppress(".") + pp.MatchFirst([kw("print"), kw("println")]) + LP + expr + RP + SEMI)
impresion_simple = (pp.Suppress(kw("print")) + LP + expr + RP + SEMI).set_parse_action(lambda t: ["print"] + list(t))
impresion = pp.MatchFirst([impresion_java, impresion_simple])
def pa_print(t):
    e = valor(t[1]); return Nodo([t[0]] + (e if isinstance(e, list) else [e]))
impresion.set_parse_action(pa_print)
break_stmt = (pp.Suppress(kw("break")) + SEMI).set_parse_action(lambda: Nodo(["break"]))
continue_stmt = (pp.Suppress(kw("continue")) + SEMI).set_parse_action(lambda: Nodo(["continue"]))
bloque <<= (LB + pp.Group(pp.ZeroOrMore(sentencia)) + RB).set_parse_action(lambda t: Nodo(lista(t[0])))
if_stmt = pp.Forward()
else_stmt = pp.Forward()
else_stmt <<= pp.MatchFirst([if_stmt, bloque])
def pa_if(t):
    r = ["if", valor(t[0]), valor(t[1])]
    if len(t) > 2: r.append(valor(t[2]))
    return Nodo(r)
if_stmt <<= (pp.Suppress(kw("if")) + LP + expr + RP + bloque + pp.Optional(pp.Suppress(kw("else")) + else_stmt)).set_parse_action(pa_if)
while_stmt = (pp.Suppress(kw("while")) + LP + expr + RP + bloque).set_parse_action(lambda t: Nodo(["while", valor(t[0]), valor(t[1])]))
do_stmt = (pp.Suppress(kw("do")) + bloque + pp.Suppress(kw("while")) + LP + expr + RP + SEMI).set_parse_action(lambda t: Nodo(["dowhile", valor(t[0]), valor(t[1])]))
inicio_decl = tipo + ident + pp.Suppress("=") + expr
inicio_asig = ident + pp.Suppress("=") + expr
inicio = pp.MatchFirst([inicio_decl, inicio_asig])
paso = pp.MatchFirst([
    ident + pp.MatchFirst([pp.Literal("++"), pp.Literal("--")]),
    ident + op_asig + expr,
    ident + pp.Suppress("=") + expr,
])
for_stmt = (pp.Suppress(kw("for")) + LP + pp.Group(inicio) + SEMI + expr + SEMI + pp.Optional(pp.Group(paso)) + RP + bloque)
def pa_for(t):
    ini = lista(t[0])
    if len(ini) == 3: tipo_v, var, inicial = ini
    else: tipo_v, var, inicial = None, ini[0], ini[1]
    cond = valor(t[1])
    if len(t) == 4:
        p = valor(t[2])
        if len(p) == 2: paso_v = p
        else:
            e = p[2]; paso_v = [p[0], p[1]] + (e if isinstance(e, list) else [e])
    else:
        op = next((x for x in (cond if isinstance(cond, list) else []) if x in ("<", "<=", ">", ">=")), "<")
        paso_v = [var, "-1" if op in (">", ">=") else "1"]
    cab = ([tipo_v] if tipo_v else []) + [var, valor(inicial), cond, paso_v]
    return Nodo(["for", cab, valor(t[-1])])
for_stmt.set_parse_action(pa_for)
literal_case = pp.MatchFirst([numero, string_lit, char_lit, boolean_lit])
case_stmt = (pp.Suppress(kw("case")) + literal_case + pp.Optional(pp.Literal(":")) + pp.Group(pp.ZeroOrMore(~(kw("case") | kw("default") | RB) + sentencia)))
def pa_case(s, loc, t):
    linea = pp.lineno(loc, s)
    if str(t[0]) == '"__CASE_COLON_MISSING__"':
        ERRORES.append((linea, "case", "faltan dos puntos"))
        return Nodo(["case", ["error", "case", "faltan dos puntos"], lista(t[-1])])
    if len(t) >= 3 and t[1] == ":":
        return Nodo(["case", t[0], lista(t[2])])
    ERRORES.append((linea, "case", "faltan dos puntos"))
    return Nodo(["case", ["error", "case", "faltan dos puntos"], lista(t[-1])])
case_stmt.set_parse_action(pa_case)
default_stmt = (pp.Suppress(kw("default")) + COLON + pp.Group(pp.ZeroOrMore(~RB + sentencia))).set_parse_action(lambda t: Nodo(["default", lista(t[0])]))
switch_stmt = (pp.Suppress(kw("switch")) + LP + expr + RP + LB + pp.Group(pp.ZeroOrMore(case_stmt)) + pp.Optional(default_stmt) + RB).set_parse_action(lambda t: Nodo(["switch", valor(t[0]), valor(t[1]) + ([valor(t[2])] if len(t) > 2 else [])]))
ERRORES = []

def analizar_cabecera_control(s, loc, t, nombre):
    encabezado = t[0]
    cuerpo = valor(t[1])
    linea = pp.lineno(loc, s)
    contenido = encabezado[encabezado.find("(") + 1:]
    tiene_cierre = ")" in contenido
    if tiene_cierre:
        contenido = contenido[:contenido.rfind(")")].strip()
    else:
        contenido = contenido.strip()
    if nombre == "for":
        if not tiene_cierre:
            contexto, desc = "for", "falta parentesis de cierre"
        elif contenido.count(";") != 2:
            contexto, desc = "for", "faltan separadores ';' en el encabezado"
        else:
            contexto, desc = "for", "encabezado mal formado"
        nodo = ["error", contexto, desc]
        ERRORES.append((linea, contexto, desc))
        return Nodo(["for", nodo, cuerpo])
    if not tiene_cierre:
        contexto, desc = nombre, "falta parentesis de cierre"
        nodo = ["error", contexto, desc]
    elif not contenido:
        contexto, desc = "condicion", "condicion vacia"
        nodo = ["error", contexto, desc]
    else:
        try:
            expr.parse_string(contenido, parse_all=True)
            contexto, desc = nombre, "encabezado mal formado"
        except pp.ParseBaseException:
            contexto, desc = "condicion", "expresion incompleta"
        nodo = ["error", contexto, desc]
    ERRORES.append((linea, contexto, desc))
    salida = [nombre, nodo, cuerpo]
    if nombre == "if" and len(t) > 2:
        salida.append(valor(t[2]))
    return Nodo(salida)

cabecera_if_mala = pp.Regex(r"if\s*\([^{}]*?(?:\)|(?=\{))\s*(?=\{)") + bloque + pp.Optional(pp.Suppress(kw("else")) + else_stmt)
cabecera_if_mala.set_parse_action(lambda s, l, t: analizar_cabecera_control(s, l, t, "if"))
cabecera_while_mala = pp.Regex(r"while\s*\([^{}]*?(?:\)|(?=\{))\s*(?=\{)") + bloque
cabecera_while_mala.set_parse_action(lambda s, l, t: analizar_cabecera_control(s, l, t, "while"))
cabecera_for_mala = pp.Regex(r"for\s*\([^{}]*?(?:\)|(?=\{))\s*(?=\{)") + bloque
cabecera_for_mala.set_parse_action(lambda s, l, t: analizar_cabecera_control(s, l, t, "for"))
do_malo = pp.Regex(r"do\s*\{[\s\S]*?\}\s*while\s*\([^{}]*\)")
def pa_do_malo(s, loc, t):
    bruto = t[0]
    linea = pp.lineno(loc, s)
    cuerpo_txt = re.search(r"do\s*\{([\s\S]*?)\}\s*while", bruto).group(1)
    cuerpo_normalizado = re.sub(r"(?m)(\b[A-Za-z][A-Za-z0-9]*\s*(?:\+\+|--))\s*$", r"\1;", cuerpo_txt)
    try:
        cuerpo_resultado = pp.Group(pp.ZeroOrMore(sentencia)).parse_string(cuerpo_normalizado, parse_all=True)
        cuerpo = lista(cuerpo_resultado[0])
    except pp.ParseBaseException:
        cuerpo = []
    ERRORES.append((linea, "dowhile", "falta punto y coma"))
    condicion_txt = re.search(r"while\s*\((.*?)\)", bruto, re.S).group(1)
    try:
        condicion = valor(expr.parse_string(condicion_txt, parse_all=True)[0])
    except pp.ParseBaseException:
        condicion = ["error", "condicion", "expresion incompleta"]
    return Nodo(["dowhile", cuerpo, condicion])
do_malo.set_parse_action(pa_do_malo)

def pa_error(s, loc, t):
    bruto = t[0].strip(); linea = pp.lineno(loc, s)
    if bruto.rstrip(";") == "__MISSING_BLOCK__":
        ERRORES.append((linea, "bloque", "falta llave de cierre"))
        return Nodo(["error", "bloque", "falta llave de cierre"])
    if not bruto: return Nodo(["error", "sentencia", "sentencia no reconocida"])
    if bruto.startswith("System.out.print"):
        contexto, desc = "print", ("falta parentesis de cierre" if bruto.count("(") > bruto.count(")") else "falta punto y coma")
    elif bruto.startswith("break"): contexto, desc = "break", "falta punto y coma"
    elif bruto.startswith("continue"): contexto, desc = "continue", "falta punto y coma"
    elif bruto.endswith(("++", "--")): contexto, desc = "incremento", "falta punto y coma"
    elif any(bruto.startswith(x + " ") for x in TIPOS):
        if re.match(r"^(?:int|float|boolean|char|String)\s+\d", bruto): contexto, desc = "sentencia", "sentencia no reconocida"
        else: contexto, desc = "declaracion", "falta punto y coma"
    elif "=" in bruto: contexto, desc = "asignacion", "falta punto y coma"
    else: contexto, desc = "sentencia", "sentencia no reconocida"
    ERRORES.append((linea, contexto, desc))
    return Nodo(["error", contexto, desc])
error_stmt = pp.Regex(r"[^\n{};]+(?:;)?").set_parse_action(pa_error)

sentencia <<= pp.MatchFirst([declaracion, asignacion, incremento, impresion, break_stmt, continue_stmt,
                             if_stmt, while_stmt, do_malo, do_stmt, for_stmt, switch_stmt, cabecera_if_mala, cabecera_while_mala, cabecera_for_mala, bloque, error_stmt])
programa = pp.ZeroOrMore(sentencia) + pp.StringEnd()

def analizar_codigo(codigo, mostrar=True):
    global ERRORES
    ERRORES = []
    codigo = codigo.lstrip("\ufeff")
    codigo = COMENTARIOS.transform_string(codigo)
    codigo = re.sub(
        r'(?m)^(\s*case\s+)(-?\d+(?:\.\d+)?|true|false|"(?:[^"\\]|\\.)*"|\'(?:[^\'\\]|\\.)*\')(\s*)(?=\n)',
        r'\1"__CASE_COLON_MISSING__":', codigo
    )
    balance = 0
    for caracter in codigo:
        if caracter == "{": balance += 1
        elif caracter == "}": balance -= 1
    if balance > 0:
        codigo = codigo.rstrip() + "\n" + "\n".join("__MISSING_BLOCK__;\n}" for _ in range(balance))
    try:
        resultado = programa.parse_string(codigo, parse_all=True)
        arbol = lista(resultado)
    except pp.ParseBaseException as exc:
        ERRORES.append((exc.lineno, "sentencia", "error sintactico no recuperado"))
        arbol = []
    if mostrar:
        if ERRORES:
            print("Errores de sintaxis")
            for linea, contexto, desc in sorted(ERRORES):
                print(f"ERROR SINTACTICO (linea {linea}): {contexto} - {desc}")
        else:
            print("Código Válido")
        print("\nÁrbol de Análisis Sintáctico")
        import pprint
        pprint.pprint(arbol, width=100, sort_dicts=False)
    return arbol, sorted(ERRORES)

def cargar_archivo():
    try:
        from google.colab import files
        subidos = files.upload()
        if not subidos:
            raise ValueError("No se selecciono ningun archivo.")
        nombre = next(iter(subidos))
        if not nombre.lower().endswith(".txt"):
            raise ValueError("El archivo debe tener extension .txt.")
        return subidos[nombre].decode("utf-8-sig")
    except ImportError:
        nombre = input("Ingrese la ruta del archivo .txt: ").strip()
        if not nombre.lower().endswith(".txt"):
            raise ValueError("El archivo debe tener extension .txt.")
        with open(nombre, "r", encoding="utf-8-sig") as archivo:
            return archivo.read()
if __name__ == "__main__":
    try:
        codigo_fuente = cargar_archivo()
        analizar_codigo(codigo_fuente)
    except Exception as error:
        print("No fue posible analizar el archivo:", error)